# 02. 모델 학습 및 개선

학습 데이터로 모델을 학습하고 검증 데이터로 설정과 판단 기준을 선택한다.
테스트 데이터의 예측·성능 평가는 03 노트북에서 진행한다.

## 1. 데이터 불러오기

원본 측정값과 검사 결과를 읽고 정상은 0, 불량은 1로 변환한다.

In [13]:
from pathlib import Path
import pandas as pd

project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

data_dir = project_dir / "data" / "raw"

# 공정 측정값
X = pd.read_csv(
    data_dir / "secom.data",
    sep=r"\s+",
    header=None
)

# 검사 결과와 검사 시각
labels = pd.read_csv(
    data_dir / "secom_labels.data",
    sep=r"\s+",
    header=None,
    names=["result", "timestamp"]
)

# 정상 0, 불량 1로 변환
y = labels["result"].map({-1: 0, 1: 1})
y.name = "is_defect"

print("측정값 크기:", X.shape)
print("정답 개수:", len(y))

측정값 크기: (1567, 590)
정답 개수: 1567


## 2. 학습·검증·테스트 데이터 분리

약 60% / 20% / 20%로 나누고 정상·불량 비율을 유지한다.
기존 실험과 같은 분할을 사용하도록 난수 기준을 42로 고정한다.

In [14]:
from sklearn.model_selection import train_test_split

# 전체 데이터의 20%를 테스트용으로 분리
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# 남은 80%의 25%를 검증용으로 분리
# 전체 기준으로 학습 60%, 검증 20%, 테스트 20%
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.25,
    stratify=y_train_val,
    random_state=42
)

split_summary = pd.DataFrame({
    "데이터 수": [len(y_train), len(y_val), len(y_test)],
    "정상 수": [
        (y_train == 0).sum(),
        (y_val == 0).sum(),
        (y_test == 0).sum()
    ],
    "불량 수": [
        y_train.sum(),
        y_val.sum(),
        y_test.sum()
    ],
    "불량 비율(%)": [
        y_train.mean() * 100,
        y_val.mean() * 100,
        y_test.mean() * 100
    ]
}, index=["학습", "검증", "테스트"])

split_summary.round(2)

,데이터 수,정상 수,불량 수,불량 비율(%)
학습,939,877,62,6.60
검증,314,293,21,6.69
테스트,314,293,21,6.69


## 3. 사용할 측정 항목 선택

학습 데이터에서 결측치 비율이 50%를 초과하거나 관측값이 한 종류 이하인 열을 제외한다.
같은 항목과 순서를 검증·테스트 데이터에도 적용한다.

In [15]:
# 학습 데이터에서만 제외 기준 계산
train_missing_ratio = X_train.isna().mean()
train_unique_count = X_train.nunique(dropna=True)

drop_mask = (
    (train_missing_ratio > 0.5)
    | (train_unique_count <= 1)
)

selected_columns = X_train.columns[~drop_mask]

# 같은 측정 항목을 세 그룹에 적용
X_train_selected = X_train.loc[:, selected_columns].copy()
X_val_selected = X_val.loc[:, selected_columns].copy()
X_test_selected = X_test.loc[:, selected_columns].copy()

print("기존 측정 항목 수:", X_train.shape[1])
print("제외한 항목 수:", drop_mask.sum())
print("남은 항목 수:", len(selected_columns))

기존 측정 항목 수: 590
제외한 항목 수: 146
남은 항목 수: 444


## 4. 결측치 채우기

학습 데이터의 항목별 중앙값으로 결측치를 채운다.
검증·테스트 데이터에도 학습 데이터에서 계산한 중앙값을 사용한다.

In [16]:
from sklearn.impute import SimpleImputer
import numpy as np

imputer = SimpleImputer(strategy="median")

# 학습 데이터에서 중앙값 계산 후 결측치 채우기
X_train_imputed = imputer.fit_transform(X_train_selected)

# 학습 데이터의 중앙값을 그대로 적용
X_val_imputed = imputer.transform(X_val_selected)
X_test_imputed = imputer.transform(X_test_selected)

print("학습 데이터 크기:", X_train_imputed.shape)
print("학습 결측치 수:", np.isnan(X_train_imputed).sum())
print("검증 결측치 수:", np.isnan(X_val_imputed).sum())
print("테스트 결측치 수:", np.isnan(X_test_imputed).sum())

학습 데이터 크기: (939, 444)
학습 결측치 수: 0
검증 결측치 수: 0
테스트 결측치 수: 0


## 5. 측정값 표준화

로지스틱 회귀에 사용할 측정값의 숫자 규모를 맞춘다.
학습 데이터의 평균과 표준편차를 검증·테스트 데이터에도 적용한다.

In [17]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# 학습 데이터에서 기준 계산 후 표준화
X_train_scaled = scaler.fit_transform(X_train_imputed)

# 같은 기준으로 검증·테스트 데이터 변환
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("학습 데이터 크기:", X_train_scaled.shape)
print("검증 데이터 크기:", X_val_scaled.shape)
print("테스트 데이터 크기:", X_test_scaled.shape)

print("첫 번째 항목의 학습 평균:",
      round(X_train_scaled[:, 0].mean(), 6))
print("첫 번째 항목의 학습 표준편차:",
      round(X_train_scaled[:, 0].std(), 6))

학습 데이터 크기: (939, 444)
검증 데이터 크기: (314, 444)
테스트 데이터 크기: (314, 444)
첫 번째 항목의 학습 평균: 0.0
첫 번째 항목의 학습 표준편차: 1.0


## 6. 로지스틱 회귀 학습

표준화한 학습 데이터로 기준 모델을 학습한다.
불량 사례가 적으므로 클래스별 가중치를 적용한다.

In [18]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=2000,
    random_state=42
)

# 측정값과 정답으로 모델 학습
logistic_model.fit(X_train_scaled, y_train)

print("로지스틱 회귀 학습 완료")
print("학습한 측정 항목 수:", logistic_model.n_features_in_)
print("정답 클래스:", logistic_model.classes_)

로지스틱 회귀 학습 완료
학습한 측정 항목 수: 444
정답 클래스: [0 1]


## 7. 로지스틱 회귀 검증

검증 데이터에서 정확도와 불량 Precision·Recall·F1을 확인한다.
이 단계에서는 기본 판단 기준 0.5를 사용한다.

In [19]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# 검증 데이터의 정상·불량 예측
logistic_val_pred = logistic_model.predict(X_val_scaled)

# 불량(1)을 기준으로 성능 계산
logistic_val_metrics = {
    "정확도": accuracy_score(y_val, logistic_val_pred),
    "불량 Precision": precision_score(
        y_val, logistic_val_pred, zero_division=0
    ),
    "불량 Recall": recall_score(
        y_val, logistic_val_pred, zero_division=0
    ),
    "불량 F1": f1_score(
        y_val, logistic_val_pred, zero_division=0
    )
}

pd.DataFrame(
    [logistic_val_metrics],
    index=["로지스틱 회귀"]
).round(4)

,정확도,불량 Precision,불량 Recall,불량 F1
로지스틱 회귀,0.8662,0.1613,0.2381,0.1923


## 8. 기본 Random Forest 학습 및 검증

결측치를 채운 데이터로 Random Forest를 학습한다.
트리 모델에는 표준화 전 데이터를 사용하며 기본 판단 기준 0.5로 비교한다.

In [20]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

# 결측치를 채운 데이터로 학습
forest_model.fit(X_train_imputed, y_train)

# 검증 데이터 예측
forest_val_pred = forest_model.predict(X_val_imputed)

forest_val_metrics = {
    "정확도": accuracy_score(y_val, forest_val_pred),
    "불량 Precision": precision_score(
        y_val, forest_val_pred, zero_division=0
    ),
    "불량 Recall": recall_score(
        y_val, forest_val_pred, zero_division=0
    ),
    "불량 F1": f1_score(
        y_val, forest_val_pred, zero_division=0
    )
}

# 두 모델의 검증 성능 비교
model_comparison = pd.DataFrame(
    [logistic_val_metrics, forest_val_metrics],
    index=["로지스틱 회귀", "Random Forest"]
)

model_comparison.round(4)

,정확도,불량 Precision,불량 Recall,불량 F1
로지스틱 회귀,0.8662,0.1613,0.2381,0.1923
Random Forest,0.9331,0.0000,0.0000,0.0000


## 9. 기본 모델의 판단 기준 비교

두 모델에 판단 기준 0.1~0.5를 적용한다.
검증 데이터에서 불량 탐지·놓침과 정상 오탐을 함께 확인한다.

In [21]:
from sklearn.metrics import confusion_matrix

# 각 모델이 계산한 불량(클래스 1) 확률
logistic_val_prob = logistic_model.predict_proba(X_val_scaled)[:, 1]
forest_val_prob = forest_model.predict_proba(X_val_imputed)[:, 1]

results = []

for model_name, probabilities in [
    ("로지스틱 회귀", logistic_val_prob),
    ("Random Forest", forest_val_prob)
]:
    for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
        predictions = (probabilities >= threshold).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_val, predictions, labels=[0, 1]
        ).ravel()

        results.append({
            "모델": model_name,
            "판단 기준": threshold,
            "불량 Precision": precision_score(
                y_val, predictions, zero_division=0
            ),
            "불량 Recall": recall_score(
                y_val, predictions, zero_division=0
            ),
            "불량 F1": f1_score(
                y_val, predictions, zero_division=0
            ),
            "불량 탐지": tp,
            "불량 놓침": fn,
            "정상 오탐": fp
        })

threshold_comparison = pd.DataFrame(results)
threshold_comparison.round(4)

,모델,판단 기준,불량 Precision,불량 Recall,불량 F1,불량 탐지,불량 놓침,정상 오탐
0,로지스틱 회귀,0.1,0.0877,0.2381,0.1282,5,16,52
1,로지스틱 회귀,0.2,0.1064,0.2381,0.1471,5,16,42
2,로지스틱 회귀,0.3,0.1190,0.2381,0.1587,5,16,37
3,로지스틱 회귀,0.4,0.1429,0.2381,0.1786,5,16,30
4,로지스틱 회귀,0.5,0.1613,0.2381,0.1923,5,16,26
5,Random Forest,0.1,0.0844,0.9048,0.1545,19,2,206
6,Random Forest,0.2,0.1266,0.4762,0.2000,10,11,69
7,Random Forest,0.3,0.2000,0.1905,0.1951,4,17,16
8,Random Forest,0.4,0.4000,0.0952,0.1538,2,19,3
9,Random Forest,0.5,0.0000,0.0000,0.0000,0,21,0


## 10. Random Forest 설정 비교

최대 깊이 3가지와 최소 노드 데이터 수 3가지를 조합해 비교한다.
각 설정에서 판단 기준 0.1~0.5를 검증한다. 기존 설정도 비교에 포함한다.
깊이는 트리의 복잡도를, 최소 노드 데이터 수는 마지막 노드에 필요한 최소 표본 수를 뜻한다.

In [22]:
experiment_models = {}
experiment_results = []

for max_depth in [None, 5, 10]:
    for min_samples_leaf in [2, 5, 10]:
        depth_label = "제한 없음" if max_depth is None else str(max_depth)
        setting = (depth_label, min_samples_leaf)

        # 기존 설정은 이미 학습한 모델을 재사용
        if max_depth is None and min_samples_leaf == 2:
            model = forest_model
        else:
            model = RandomForestClassifier(
                n_estimators=200,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
            model.fit(X_train_imputed, y_train)

        experiment_models[setting] = model
        defect_column = list(model.classes_).index(1)
        probabilities = model.predict_proba(X_val_imputed)[:, defect_column]

        for threshold in [0.1, 0.2, 0.3, 0.4, 0.5]:
            predictions = (probabilities >= threshold).astype(int)
            tn, fp, fn, tp = confusion_matrix(
                y_val, predictions, labels=[0, 1]
            ).ravel()

            experiment_results.append({
                "모델": "Random Forest",
                "최대 깊이": depth_label,
                "최소 노드 데이터 수": min_samples_leaf,
                "판단 기준": threshold,
                "불량 Precision": precision_score(y_val, predictions, zero_division=0),
                "불량 Recall": recall_score(y_val, predictions, zero_division=0),
                "불량 F1": f1_score(y_val, predictions, zero_division=0),
                "불량 탐지": tp,
                "불량 놓침": fn,
                "정상 오탐": fp
            })

        print(f"검증 완료: 최대 깊이 {depth_label}, 최소 노드 데이터 수 {min_samples_leaf}")

experiment_comparison = pd.DataFrame(experiment_results)
experiment_comparison.sort_values(
    "불량 F1", ascending=False, kind="stable"
).head(10).round(4)

검증 완료: 최대 깊이 제한 없음, 최소 노드 데이터 수 2
검증 완료: 최대 깊이 제한 없음, 최소 노드 데이터 수 5
검증 완료: 최대 깊이 제한 없음, 최소 노드 데이터 수 10
검증 완료: 최대 깊이 5, 최소 노드 데이터 수 2
검증 완료: 최대 깊이 5, 최소 노드 데이터 수 5
검증 완료: 최대 깊이 5, 최소 노드 데이터 수 10
검증 완료: 최대 깊이 10, 최소 노드 데이터 수 2
검증 완료: 최대 깊이 10, 최소 노드 데이터 수 5
검증 완료: 최대 깊이 10, 최소 노드 데이터 수 10


,모델,최대 깊이,최소 노드 데이터 수,판단 기준,불량 Precision,불량 Recall,불량 F1,불량 탐지,불량 놓침,정상 오탐
32,Random Forest,10,2,0.3,0.2000,0.2857,0.2353,6,15,24
28,Random Forest,5,10,0.4,0.1429,0.5238,0.2245,11,10,66
12,Random Forest,제한 없음,10,0.3,0.1204,0.6190,0.2016,13,8,95
42,Random Forest,10,10,0.3,0.1204,0.6190,0.2016,13,8,95
1,Random Forest,제한 없음,2,0.2,0.1266,0.4762,0.2000,10,11,69
2,Random Forest,제한 없음,2,0.3,0.2000,0.1905,0.1951,4,17,16
31,Random Forest,10,2,0.2,0.1111,0.5238,0.1833,11,10,88
43,Random Forest,10,10,0.4,0.1304,0.2857,0.1791,6,15,40
22,Random Forest,5,5,0.3,0.1020,0.7143,0.1786,15,6,132
13,Random Forest,제한 없음,10,0.4,0.1277,0.2857,0.1765,6,15,41


## 11. 기본 모델과 개선 후보 비교

로지스틱 회귀와 Random Forest의 전체 후보를 검증 F1 순으로 비교한다.
F1이 같으면 Recall, 그다음 Precision이 높은 후보를 우선한다.
기본 모델과 설정 변경 후보의 최고 검증 성능도 함께 확인한다.

In [23]:
# 로지스틱 회귀의 판단 기준별 결과
logistic_candidates = threshold_comparison[
    threshold_comparison["모델"] == "로지스틱 회귀"
].copy()
logistic_candidates["최대 깊이"] = "해당 없음"
logistic_candidates["최소 노드 데이터 수"] = 0

# 기본 Random Forest도 experiment_comparison에 포함되어 있음
all_candidates = pd.concat(
    [logistic_candidates, experiment_comparison], ignore_index=True
)
ranked_candidates = all_candidates.sort_values(
    ["불량 F1", "불량 Recall", "불량 Precision"],
    ascending=[False, False, False],
    kind="stable"
).reset_index(drop=True)

comparison_columns = [
    "모델", "최대 깊이", "최소 노드 데이터 수", "판단 기준",
    "불량 Precision", "불량 Recall", "불량 F1",
    "불량 탐지", "불량 놓침", "정상 오탐"
]

# 각 그룹의 최고 후보만 비교
basic_forest_mask = (
    (ranked_candidates["모델"] == "Random Forest")
    & (ranked_candidates["최대 깊이"] == "제한 없음")
    & (ranked_candidates["최소 노드 데이터 수"] == 2)
)
logistic_mask = ranked_candidates["모델"] == "로지스틱 회귀"
changed_forest_mask = (
    (ranked_candidates["모델"] == "Random Forest") & ~basic_forest_mask
)

best_by_group = pd.concat([
    ranked_candidates[logistic_mask].head(1).assign(구분="로지스틱 회귀"),
    ranked_candidates[basic_forest_mask].head(1).assign(구분="기본 Random Forest"),
    ranked_candidates[changed_forest_mask].head(1).assign(구분="설정 변경 Random Forest")
], ignore_index=True)

best_by_group[["구분"] + comparison_columns].round(4)

,구분,모델,최대 깊이,최소 노드 데이터 수,판단 기준,불량 Precision,불량 Recall,불량 F1,불량 탐지,불량 놓침,정상 오탐
0,로지스틱 회귀,로지스틱 회귀,해당 없음,0,0.5,0.1613,0.2381,0.1923,5,16,26
1,기본 Random Forest,Random Forest,제한 없음,2,0.2,0.1266,0.4762,0.2000,10,11,69
2,설정 변경 Random Forest,Random Forest,10,2,0.3,0.2000,0.2857,0.2353,6,15,24


## 12. 측정 항목 수를 줄여 학습·비교

학습 데이터로 학습한 기본 Random Forest의 중요도를 기준으로
측정 항목을 50개·100개·200개로 줄여 비교한다.

항목 수의 영향을 확인하기 위해 모델 설정과 판단 기준 0.2를 동일하게 유지한다.
전체 444개 항목을 사용하는 기본 모델도 함께 비교한다.

In [26]:
# 학습 데이터로 학습한 기본 모델의 항목 중요도
feature_importance = pd.Series(
    forest_model.feature_importances_,
    index=selected_columns
).sort_values(ascending=False, kind="stable")

feature_count_models = {}
feature_count_columns = {}
feature_count_results = []

for feature_count in [50, 100, 200, len(selected_columns)]:
    if feature_count == len(selected_columns):
        # 전체 항목은 기존 모델과 입력 순서 그대로 사용
        columns = selected_columns.tolist()
        model = forest_model
    else:
        # 중요도가 높은 항목 선택
        columns = feature_importance.head(feature_count).index.tolist()

        model = RandomForestClassifier(
            n_estimators=200,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        )

    # 선택한 열이 전처리 배열에서 몇 번째에 있는지 확인
    positions = selected_columns.get_indexer(columns)

    train_input = X_train_imputed[:, positions]
    val_input = X_val_imputed[:, positions]

    if feature_count != len(selected_columns):
        model.fit(train_input, y_train)

    feature_count_models[feature_count] = model
    feature_count_columns[feature_count] = columns

    defect_column = list(model.classes_).index(1)
    probabilities = model.predict_proba(val_input)[:, defect_column]

    # 모든 모델에 같은 판단 기준 적용
    predictions = (probabilities >= 0.2).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_val, predictions, labels=[0, 1]
    ).ravel()

    feature_count_results.append({
        "측정 항목 수": feature_count,
        "판단 기준": 0.2,
        "불량 Precision": precision_score(
            y_val, predictions, zero_division=0
        ),
        "불량 Recall": recall_score(
            y_val, predictions, zero_division=0
        ),
        "불량 F1": f1_score(
            y_val, predictions, zero_division=0
        ),
        "불량 탐지": tp,
        "불량 놓침": fn,
        "정상 오탐": fp
    })

    print(f"검증 완료: 측정 항목 {feature_count}개")

feature_count_comparison = pd.DataFrame(feature_count_results)
feature_count_comparison.round(4)

검증 완료: 측정 항목 50개
검증 완료: 측정 항목 100개
검증 완료: 측정 항목 200개
검증 완료: 측정 항목 444개


,측정 항목 수,판단 기준,불량 Precision,불량 Recall,불량 F1,불량 탐지,불량 놓침,정상 오탐
0,50,0.2,0.1146,0.5238,0.1880,11,10,85
1,100,0.2,0.1279,0.5238,0.2056,11,10,75
2,200,0.2,0.1236,0.5238,0.2000,11,10,78
3,444,0.2,0.1266,0.4762,0.2000,10,11,69


100개 항목을 사용했을 때 기본 설정 대비 검증 F1이
0.2000에서 0.2056으로 소폭 상승했지만 정상 오탐도 증가했다.

앞선 설정 비교에서 선택한 모델의 검증 F1(0.2353)보다 낮아,
최종 모델에는 항목 축소를 적용하지 않는다.

## 13. 모델과 판단 기준 선택

항목 축소 실험은 채택하지 않고,
11번에서 비교한 전체 항목 사용 후보 중 검증 F1이 가장 높은 설정을 선택한다.
모델 선택에는 테스트 성능을 사용하지 않는다.

In [27]:
best_candidate = ranked_candidates.iloc[0]
selected_model_name = best_candidate["모델"]
selected_threshold = float(best_candidate["판단 기준"])

if selected_model_name == "로지스틱 회귀":
    selected_model = logistic_model
    selected_scaler = scaler
else:
    selected_setting = (
        best_candidate["최대 깊이"],
        int(best_candidate["최소 노드 데이터 수"])
    )
    selected_model = experiment_models[selected_setting]
    selected_scaler = None

print("선택 모델:", selected_model_name)
print("불량 판단 기준:", selected_threshold)
print("검증 불량 F1:", round(float(best_candidate["불량 F1"]), 4))
pd.DataFrame([best_candidate])[comparison_columns].round(4)

선택 모델: Random Forest
불량 판단 기준: 0.3
검증 불량 F1: 0.2353


,모델,최대 깊이,최소 노드 데이터 수,판단 기준,불량 Precision,불량 Recall,불량 F1,불량 탐지,불량 놓침,정상 오탐
0,Random Forest,10,2,0.3,0.2,0.2857,0.2353,6,15,24


## 14. 모델과 전처리 정보 저장

선택한 모델, 결측치 처리 도구, 표준화 도구, 측정 항목과 판단 기준을 저장한다.
03에서 같은 테스트 데이터를 사용하도록 행 번호도 함께 저장한다.
이 셀을 실행하면 기존 `models/defect_prediction.joblib`를 갱신한다.

In [30]:
import joblib
import sklearn

# 프로젝트 최상위에 models 폴더 생성
model_dir = project_dir / "models"
model_dir.mkdir(exist_ok=True)

# 예측과 최종 평가에 필요한 정보를 하나로 묶기
model_bundle = {
    "model_name": f"{selected_model_name} (두 번째 모델)",
    "model": selected_model,
    "imputer": imputer,
    "scaler": selected_scaler,
    "selected_columns": selected_columns.tolist(),
    "threshold": selected_threshold,
    "test_indices": X_test.index.tolist(),
    "sklearn_version": sklearn.__version__,
    "validation_selection": best_candidate.to_dict(),
    "split_random_state": 42
}

model_path = model_dir / "defect_prediction.joblib"

joblib.dump(model_bundle, model_path)

print("저장 파일:", model_path)
print("파일 존재:", model_path.exists())
print("저장 모델:", model_bundle["model_name"])
print("판단 기준:", model_bundle["threshold"])
print("측정 항목 수:", len(model_bundle["selected_columns"]))

저장 파일: c:\Workspace\manufacturing-defect-prediction\models\defect_prediction.joblib
파일 존재: True
저장 모델: Random Forest (두 번째 모델)
판단 기준: 0.3
측정 항목 수: 444


### 첫 번째 모델 저장

전체 444개 항목으로 학습한 기본 Random Forest와
처음 선택한 판단 기준 0.2를 별도 파일로 저장한다.

In [31]:
# 전처리 정보와 테스트 행 번호는 동일하게 유지
baseline_bundle = model_bundle.copy()

baseline_bundle.update({
    "model_name": "Random Forest (첫 번째 모델)",
    "model": forest_model,
    "scaler": None,
    "threshold": 0.2
})

# 설정 변경 모델의 선택 정보는 제외
baseline_bundle.pop("validation_selection", None)

baseline_path = model_dir / "defect_prediction_baseline.joblib"
joblib.dump(baseline_bundle, baseline_path)

print("첫 번째 모델 저장:", baseline_path)
print("파일 존재:", baseline_path.exists())

첫 번째 모델 저장: c:\Workspace\manufacturing-defect-prediction\models\defect_prediction_baseline.joblib
파일 존재: True


## 15. 실험 범위와 평가 주의사항

Random Forest의 최대 깊이와 최소 노드 데이터 수를 조합한
9개 설정에 판단 기준 0.1~0.5를 적용해 검증 성능을 비교했다.

추가로 기본 Random Forest 설정과 판단 기준 0.2를 고정하여
측정 항목 수 50개·100개·200개·444개를 비교했다.
항목 축소 실험은 채택하지 않고, 전체 444개 항목을 유지했다.

검증 데이터의 불량이 21건으로 적고 같은 검증 데이터를 여러 후보 비교에
사용했으므로, 선택한 설정이 다른 데이터에서도 더 좋다고 보장할 수 없다.

첫 번째 모델의 테스트 결과를 확인한 뒤 개선 실험을 추가했다.
설정 선택에는 학습·검증 데이터만 사용했으며,
03 노트북의 결과는 같은 테스트 데이터의 재평가로 기록한다.